[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c03_rules.ipynb)

# Why we write dy/dx: the product rule and the chain rule

In November 1675 Gottfried Leibniz asked his notebook whether d(xy) is the same thing as dx dy, and found that it
is not. Here you run his question on a tea stall's takings, chain two rates together, and check any derivative with
a tiny window.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
▶ Run button) to run it. Then run each cell in turn, from top to bottom. When a slider appears, drag it. Change any
number you like and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in these notebooks. Run this cell first.
import math, warnings
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the curve · slopes and rates · areas

warnings.filterwarnings("ignore", message=".*as float was deprecated.*")   # a browser-only notice, harmless
plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

## 1 · The tea stall

A stall sells 200 cups a day at \$3.00 each: \$600 a day. Next month the price changes and so do the sales. The
change in the takings splits into three pieces: the price change on the cups you already sold, the new cups at the
old price, and the **corner piece**, the price change on the new cups.

Leibniz's first guess, d(xy) = dx · dy, keeps only the corner piece.

In [ ]:
def tea_stall(price_change=0.10, cups_change=10):
    p, n = 3.00, 200
    a, b, corner = n * price_change, p * cups_change, price_change * cups_change
    total = (p + price_change) * (n + cups_change) - p * n
    fig, ax = plt.subplots(figsize=(9, 3.6))
    names = ["higher price, old cups", "new cups, old price", "the corner piece"]
    ax.barh(names[::-1], [corner, b, a], color=[CORAL, SAGE, SAGE], height=0.55)
    for y, v in enumerate([corner, b, a]):
        ax.text(v + 0.4, y, f"${v:.2f}", va="center", color=CHALK, fontsize=13)
    ax.set_xlim(0, 75); ax.set_xlabel("dollars a day"); ax.set_axisbelow(True)
    ax.set_title(f"the takings grow by ${total:.2f}: three pieces")
    plt.show()
    print(f"price ${p:.2f} → ${p + price_change:.2f},  cups {n} → {n + cups_change}")
    print(f"  higher price on the old cups:  {n} × {price_change:.2f} = ${a:8.2f}")
    print(f"  new cups at the old price:     {p:.2f} × {cups_change} = ${b:8.2f}")
    print(f"  the corner piece:              {price_change:.2f} × {cups_change} = ${corner:8.2f}")
    print(f"  total change in takings:                      ${total:8.2f}")
    print(f"  the corner's share of it: {100 * corner / total:.1f}%" if total else "")

interact(tea_stall, price_change=FloatSlider(value=0.10, min=0.01, max=0.20, step=0.01, description="price +$"),
         cups_change=IntSlider(value=10, min=1, max=20, description="cups +"));

**Notice.** Make both changes small. The first two pieces shrink in step with the changes, but the corner piece
shrinks far faster: it is a small change times a small change. In the limit it vanishes, and what is left is the
product rule: **d(uv) = u dv + v du**.

## 2 · Chained rates

A climber gains 0.3 km of height an hour, and the air cools by about 6.5 °C for every km of height. Degrees per km,
times km per hour, gives degrees per hour: dT/dt = dT/dh · dh/dt.

In [ ]:
cooling_per_km = 6.5        # °C per km (the standard atmosphere, about)
climb_km_per_h = 0.3        # km per hour
print(f"{cooling_per_km} °C per km × {climb_km_per_h} km per hour = {cooling_per_km * climb_km_per_h:.2f} °C per hour")

## 3 · Check any derivative with a tiny window

Run the cell, then type a function in the first box and your derivative in the second, using `x`, `**` for
powers, and `sin`, `cos`, `exp`, `log`, `sqrt`; press **Enter** after each. The cell measures the slope with a
tiny window at x = 1 and compares it with your answer.

In [ ]:
def check(formula, answer, a=1.0, h=1e-6):
    names = {k: getattr(np, k) for k in ("sin", "cos", "tan", "exp", "log", "sqrt", "pi")}
    f = lambda x: eval(formula, {"__builtins__": {}}, dict(names, x=x))
    g = lambda x: eval(answer, {"__builtins__": {}}, dict(names, x=x))
    window = (f(a + h) - f(a - h)) / (2 * h)
    yours = g(a)
    print(f"f(x) = {formula}")
    print(f"window slope at x = {a:g}:  {window:.4f}")
    print(f"your answer {answer} at x = {a:g}:  {yours:.4f}")
    print("they agree ✓" if abs(window - yours) < 1e-4 else "they disagree ✗: look again")

from ipywidgets import Text, fixed
interact(check, formula=Text(value="x**2", description="f(x) =", continuous_update=False),
         answer=Text(value="2*x", description="f′(x) =", continuous_update=False), a=FloatSlider(
             value=1.0, min=-3, max=3, step=0.5, description="at x ="), h=fixed(1e-6));

**Try it.** Type `sin(x**2)` with the answer `cos(x**2)`: the window says 1.0806, the answer says 0.5403, half of
it. The chain rule's missing piece is the inner change, `2*x`. Then try the video's other examples:
`x**2*sin(x)` and `x/(x**2 + 1)`.

## Sources
G. W. Leibniz, manuscripts of October–November 1675, translated by J. M. Child, *The Early Mathematical
Manuscripts of Leibniz* (1920). NASA Glenn Research Center, the standard atmosphere (lapse rate about 6.5 °C per
km). The tea-stall numbers are our own example. Retrieved 2026-10-24.